Este proyecto pretende detectar si una reclamacion de accidente vehicular es fraude o no

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from pycaret.classification import setup, compare_models, predict_model, pull
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score, roc_curve, precision_recall_curve

try:
    from xgboost import XGBClassifier
    xgb_ok = True
except ImportError:
    xgb_ok = False

warnings.filterwarnings('ignore')
import sklearn
import xgboost as xgb
import imblearn
import warnings
warnings.filterwarnings('ignore')


In [ ]:
df = pd.read_csv('fraud_oracle.csv')

### Diccionario de Datos

| Variable | Tipo | ¿Qué significa? |
| :--- | :--- | :--- |
| Month | Texto | Mes en el que ocurrió el accidente. |
| WeekOfMonth | Número | Semana del mes en la que ocurrió el accidente. |
| DayOfWeek | Texto | Día de la semana en el que ocurrió el accidente. |
| Make | Texto | Marca del vehículo involucrado. |
| AccidentArea | Texto | Tipo de zona donde ocurrió el accidente (Urbano o Rural). |
| DayOfWeekClaimed | Texto | Día de la semana en el que se realizó la reclamación. |
| MonthClaimed | Texto | Mes en el que se realizó la reclamación. |
| WeekOfMonthClaimed | Número | Semana del mes en la que se realizó la reclamación. |
| Sex | Texto | Género del conductor involucrado. |
| MaritalStatus | Texto | Estado civil del conductor. |
| Age | Número | Edad del conductor involucrado  |
| Fault | Texto | Quién fue determinado como culpable (Titular de la póliza o Tercero). |
| PolicyType | Texto | Combinación de categoría de vehículo y tipo de cobertura. |
| VehicleCategory | Texto | Categoría del vehículo (Sedán, Deportivo, Utilitario). |
| VehiclePrice | Texto (Rango) | Rango de precio del vehículo asegurado. |
| FraudFound_P | Binario (0/1) | Indica si se detectó fraude en la reclamación (1: Sí, 0: No). |
| PolicyNumber | Número | Identificador único de la póliza. |
| RepNumber | Número | Identificador del representante que procesó el caso. |
| Deductible | Número | Monto del deducible de la póliza. |
| DriverRating | Número | Calificación del conductor (1 a 4). |
| Days_Policy_Accident | Texto (Rango) | Tiempo transcurrido desde el inicio de la póliza hasta el accidente. |
| Days_Policy_Claim | Texto (Rango) | Tiempo transcurrido desde el inicio de la póliza hasta la reclamación. |
| PastNumberOfClaims | Texto (Rango) | Número de reclamaciones realizadas anteriormente por el asegurado. |
| AgeOfVehicle | Texto (Rango) | Antigüedad del vehículo involucrado. |
| AgeOfPolicyHolder | Texto (Rango) | Rango de edad del titular de la póliza. |
| PoliceReportFiled | Texto (Sí/No) | Indica si se presentó un informe policial del accidente. |
| WitnessPresent | Texto (Sí/No) | Indica si hubo testigos presentes en el accidente. |
| AgentType | Texto | Tipo de agente (Interno o Externo). |
| NumberOfSuppliments | Texto (Rango) | Cantidad de suplementos o anexos añadidos a la reclamación. |
| AddressChange_Claim | Texto | Indica si hubo un cambio de dirección durante el proceso de reclamación. |
| NumberOfCars | Texto (Rango) | Cantidad de vehículos involucrados en el incidente. |
| Year | Número | Año en el que ocurrió el accidente. |
| BasePolicy | Texto | Tipo de cobertura base (Responsabilidad Civil, Colisión, Todo Riesgo). |

In [ ]:
df.head()

In [ ]:
df.describe()


con df.describe pudimos identificar varias cosas: 
1. que el porcentaje de fraudes encontrados es 5.9% 
2. que edad tiene datos en 0 

In [ ]:
df.info()


In [ ]:
df.isnull().sum().to_string()

No hay valores nulos entonces, vamos a mirar duplicados, tipos de datos y cuales columnas si son relevantes y cuales no 

In [ ]:
df.duplicated().sum()

Tampoco hay duplicados, continuamos con investigar la columna de edad porq edad = 0 es ilogico

In [ ]:
df['Age'].unique()

miremos cuantas filas tienen edad = 0


In [ ]:
df[df['Age'] ==0]

se identifico que todas las filas de AGE que tienen "0", en "AgeOfPolicyHolder" tienen "16 to 17"	

In [ ]:
df.loc[df ['Age']== 0, 'AgeOfPolicyHolder'].unique()

Entonces voy a mirar que dato tienew AGE en las otras filas que "AgeOfPolicyHolder" tiene "16 to 17"

In [ ]:
df[df['AgeOfPolicyHolder'] == "16 to 17"][["Age", "AgeOfPolicyHolder"]]


Todos los datos de la  columna "Age" cuando  "AgeOfPolicyHolders" es igual a "16 to 17" son "0" entonces no podriamos reemplazar por algun dato existente. Continuamos revisando otras variables y ma adelante tomaremos decision sobre que hacer con esto con encontramos


Voy a verificar si en PolicyNumber hay duplicados, loq ue significaria que hay multiples reclamaciones de una poliza, en caso de que no existan duplicaciones esta columna podria eliminarse


In [ ]:
df[df.duplicated(['PolicyNumber'])]

efectivamente, no hay duplicados entonces la eliminaremos

In [ ]:
df.drop('PolicyNumber', axis = 1, inplace = True)

Ahora eliminemos las columnas que no tengan relevancia 

In [ ]:
df = df.drop(columns=[
    'MaritalStatus', 'Month', 'WeekOfMonth', 'DayOfWeek', 
    'MonthClaimed', 'WeekOfMonthClaimed', 'DayOfWeekClaimed', 
    'RepNumber', 'AgentType', 'AddressChange_Claim', 'Year','NumberOfSuppliments'])

Eliminamos las fechas tanto en las que ocurrio el accidente como en la que hicieron la reclamacion, ya que los datos que nos podrian servir son estos: Days_Policy_Accident y Days_Policy_Claim

In [ ]:
df['Days_Policy_Accident'].unique()

In [ ]:
df['Days_Policy_Claim'].unique()

Ahora tenemos dos opciones, rellenar esos "0" con 16 para que este acorde al "AgeOfPolicyholder" o graficar a traves de un boxplot para determinar si reemplazamos por la media o por la mediana

En caso de que decidamos que si vamos a convertir esos '0' a '16': 
df['Age'] = df['Age'].replace('0', '16')

Vamos a graficar con un boxplot a ver que encontramos

In [ ]:
df['Age'].plot(kind='box')
plt.show()

Encontramos que hay datos extremos o atipicos tanto "0" como mas de "75", por lo tanto vamos a reemplazar el 0 por la mediana ya que si lo hicieramos por la media los valores extremos afectarian de mal manera el resultado

In [ ]:
df.loc[df['Age'] == 0, 'Age'] = df['Age'].median()

In [ ]:
df['Age'].plot(kind='box')
plt.show()

In [ ]:

df['Age'].unique()

Como ya tenemos estos datos bien organizados, encontramos que la columna "AgePolicyOfHolder" es redudante, por lo tanto la eliminamos

In [ ]:
df = df.drop(columns=['AgeOfPolicyHolder'])

In [ ]:

df['PastNumberOfClaims'].unique()

In [ ]:
df.columns

In [ ]:
df.dtypes


pongamos en binario las variables que deban estarlo, como la de area de accidente, etc.

In [ ]:
binarizacion = {
    'Sex': {"Female": 1, "Male": 0},
    'WitnessPresent': {"Yes": 1, "No": 0},
    'AccidentArea': {"Urban": 1, "Rural": 0},
    'PoliceReportFiled': {"Yes": 1, "No": 0},
    'Fault': {'Policy Holder': 1, 'Third Party': 0}
}

for col, mapa in binarizacion.items():
    df[col] = df[col].map(mapa)

| Columna           | Valores probables           | Tipo    | Tratamiento |
| ----------------- | --------------------------- | ------- |-------------|
| Fault             | Policy Holder / Third Party | Binaria | 1/0         |
| PoliceReportFiled | Yes / No                    | Binaria | 1/0         |
| WitnessPresent    | Yes / No                    | Binaria | 1/0         |
| AccidentArea      | Urban / Rural               | Binaria | 1/0         |
| Sex               | Female / Male               | Binaria | 1/0         |


In [ ]:
df.info()

dado que las variables deben ser numericas para calcular correlaciones y para que el modelo de ML lo entienda vamos a convertirlas en numeros

In [ ]:
df['PastNumberOfClaims'].unique()

In [ ]:
df['NumberOfCars'].unique()

In [ ]:
df['VehiclePrice'].unique()

In [ ]:
df['AgeOfVehicle'].unique()

In [ ]:
df.columns = df.columns.str.strip()

df['PastNumberOfClaims'] = df['PastNumberOfClaims'].str.strip().map({
    'none': 0,
    '1': 1,            
    '2 to 4': 3,          
    'more than 4': 5
})

df['NumberOfCars'] = df['NumberOfCars'].str.strip().map({
    '1 vehicle': 1,
    '2 vehicles': 2,
    '3 to 4': 3.5,   
    '5 to 8': 6.5,
    'more than 8': 9
})

df['VehiclePrice'] = df['VehiclePrice'].str.strip().map({
    'less than 20000': 15000,
    '20000 to 29000': 24500,
    '30000 to 39000': 34500,
    '40000 to 59000': 49500,        
    '60000 to 69000': 64500,
    'more than 69000': 70000
})

df['Days_Policy_Accident'] = df['Days_Policy_Accident'].str.strip().map({
    'none': 0,
    '1 to 7': 4,
    '8 to 15': 11,              
    '15 to 30': 22,
    'more than 30': 40
})

df['Days_Policy_Claim'] = df['Days_Policy_Claim'].str.strip().map({
    'none': 0,
    '8 to 15': 11,
    '15 to 30': 22,              
    'more than 30': 40
})

df['AgeOfVehicle'] = df['AgeOfVehicle'].str.strip().map({
    'new': 1,
    '2 years': 2,
    '3 years': 3,
    '4 years': 4,
    '5 years': 5,        
    '6 years': 6,
    '7 years': 7,
    'more than 7': 10
})




In [ ]:
df.info()

Justificar estas variables


In [ ]:
df['Make'].value_counts()

In [ ]:
df['Make'] = df['Make'].replace({
    'Accura': 'Acura',
    'Nisson': 'Nissan',
    'Porche': 'Porsche',
    'Mecedes': 'Mercedes'
})

In [ ]:
df['Make'].value_counts()

In [ ]:
frec_marcas = df['Make'].value_counts().nlargest(8).index

In [ ]:
df['Make'] = df['Make'].apply(
    lambda x: x if x in frec_marcas else 'Other'
)

In [ ]:
df['Make'].value_counts()

In [ ]:
df.info()

###  Ingenieria de Caracteristicas (Perfiles de Riesgo de Fraude)

Las correlaciones individuales con FraudFound_P son debiles (~0.13 maximo), por lo que creamos variables que capturen patrones de riesgo:

In [ ]:
for col in ['Riesgo_Joven_Deportivo', 'Culpable_Auto_Caro', 'Alerta_Poliza_Nueva', 'Deducible_Relativo']:
    if col in df.columns:
        df.drop(columns=[col], inplace=True)


# 1. Sin testigos + culpa del asegurado:
df['SinTestigo_Culpable'] = ((df['WitnessPresent'] == 0) & (df['Fault'] == 1)).astype(int)

# 2. Poliza muy nueva + sin reporte policial:
# Days_Policy_Accident mapeado: 4=(1-7 dias), 11=(8-15 dias)
df['PolizaNueva_SinReporte'] = (
    (df['Days_Policy_Accident'].isin([4, 11])).astype(int) * 
    (df['PoliceReportFiled'] == 0).astype(int)
)

# 3. Conductor joven + vehiculo deportivo + culpable:
df['Score_Joven_Sport_Culpable'] = (
    (df['Age'] < 25).astype(int) + 
    (df['VehicleCategory'] == 'Sport').astype(int) + 
    (df['Fault'] == 1).astype(int)
)

# 4. Reclamacion rapida:
# Days_Policy_Claim mapeado: 0=muy rapida, 11=8-15 dias
df['Reclamacion_Rapida'] = df['Days_Policy_Claim'].isin([0, 11]).astype(int)

# 5. Vehiculo caro + deducible bajo + culpable
mediana_precio = df['VehiclePrice'].median()
df['AutoCaro_DedBajo_Culpable'] = (
    (df['VehiclePrice'] > mediana_precio).astype(int) * 
    (df['Deductible'] < 400).astype(int) * 
    (df['Fault'] == 1).astype(int)
)

# 6. Reclamador frecuente + poliza nueva
df['ReclamadorFrec_NuevaPoliza'] = (
    (df['PastNumberOfClaims'] >= 3).astype(int) * 
    (df['Days_Policy_Accident'].isin([4, 11])).astype(int)
)

# 7. Area urbana + sin testigos
df['Urbano_SinTestigo'] = ((df['AccidentArea'] == 1) & (df['WitnessPresent'] == 0)).astype(int)

# 8. Ratio tiempo reclamacion / tiempo accidente (velocidad relativa)
df['Ratio_Tiempos'] = np.where(
    df['Days_Policy_Accident'] > 0,
    df['Days_Policy_Claim'] / df['Days_Policy_Accident'],
    0
)

# 9. Score de inmediatez: poliza nueva + reclamacion rapida + vehiculo nuevo
df['Score_NuevaPoliza'] = (
    (df['Days_Policy_Accident'].isin([4, 11])).astype(int) +
    (df['Days_Policy_Claim'].isin([0, 11])).astype(int) +
    (df['AgeOfVehicle'] == 1).astype(int)
)

In [ ]:
nuevas_vars = [
    'SinTestigo_Culpable', 'PolizaNueva_SinReporte', 'Score_Joven_Sport_Culpable',
    'Reclamacion_Rapida', 'AutoCaro_DedBajo_Culpable', 'ReclamadorFrec_NuevaPoliza',
    'Urbano_SinTestigo', 'Ratio_Tiempos', 'Score_NuevaPoliza'
]

print(f"=== TASA DE FRAUDE CUANDO LA VARIABLE = 1 (vs total ~{df['FraudFound_P'].mean()*100:.1f}%) ===")
for var in nuevas_vars:
    if df[var].nunique() <= 10:
        tasa = df.groupby(var)['FraudFound_P'].mean()
        print(f"\n{var}:\n{tasa}")
    else:
        print(f"\n{var}: NoFraude={df[df['FraudFound_P']==0][var].mean():.3f}, Fraude={df[df['FraudFound_P']==1][var].mean():.3f}")

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
axes = axes.flatten()

for i, var in enumerate(nuevas_vars):
    fraud_rate = df.groupby(var)['FraudFound_P'].mean().reset_index()
    sns.barplot(data=fraud_rate, x=var, y='FraudFound_P', ax=axes[i], palette='magma')
    axes[i].set_title(f'Tasa de Fraude por {var}')
    axes[i].set_ylabel('Tasa Fraude')
    axes[i].axhline(y=df['FraudFound_P'].mean(), color='red', linestyle='--')

for j in range(i+1, len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

## Preprocesamiento para Modelado

Transformamos variables categoricas nominales en dummies con One-Hot Encoding y dividimos antes de SMOTE para evitar que haya fuga de informacion


In [ ]:


df_modelo = pd.get_dummies(df, columns=['Make', 'PolicyType', 'VehicleCategory', 'BasePolicy'], drop_first=True)

X = df_modelo.drop('FraudFound_P', axis=1)
y = df_modelo['FraudFound_P']



antes de balancear el dataser vamos a dividir el conjunto de datos en entrenamiento y prueba, cuidando de que haya el mismo 6% de fraude en ambos conjuntos

In [ ]:


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Train: {X_train.shape[0]} muestras, Fraude={y_train.mean():.4f}')
print(f'Test:  {X_test.shape[0]} muestras, Fraude={y_test.mean():.4f}')


ahora si vamos a balancear solo el conjunto de entrenamiento y escalar las variables para que en casos de que alguna variable tenga numeros muy grandes el modelo no vaya a pensar que es mas importante solo por tener un dato tan grande

In [ ]:


smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

if not isinstance(X_train_bal, pd.DataFrame):
    X_train_bal = pd.DataFrame(X_train_bal, columns=X_train.columns)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_bal)
X_test_scaled = scaler.transform(X_test)

print(f'Train bal: {X_train_bal.shape[0]} muestras, Fraude={y_train_bal.mean():.4f}')


## Entrenamiento de Modelos

ahora vamos a entrenar los modelos



lr = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
lr.fit(X_train_scaled, y_train_bal)

rf = RandomForestClassifier(n_estimators=300, random_state=42,
                             class_weight='balanced', n_jobs=-1)
rf.fit(X_train_bal, y_train_bal)

# XGBoost


xgb = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=(y_train==0).sum()/(y_train==1).sum(),
    random_state=42,
    eval_metric='logloss'
)

xgb.fit(X_train_bal, y_train_bal)

modelos = {
    'Logistic Regression': (lr, X_test_scaled),
    'Random Forest': (rf, X_test),
    'XGBoost': (xgb, X_test)
}


ahora evaluaremos los modelos

In [ ]:
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)


resultados = []

for nombre, (modelo, X_eval) in modelos.items():

    print("=" * 70)
    print(f"MODELO: {nombre}")
    print("=" * 70)



    y_pred = modelo.predict(X_eval)
    y_proba = modelo.predict_proba(X_eval)[:, 1]



    roc_auc = roc_auc_score(y_test, y_proba)
    pr_auc = average_precision_score(y_test, y_proba)

    print("\nClassification Report:\n")
    print(
        classification_report(
            y_test,
            y_pred,
            target_names=['No Fraude', 'Fraude']
        )
    )

    print(f"ROC-AUC : {roc_auc:.4f}")
    print(f"PR-AUC  : {pr_auc:.4f}")



    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(5, 4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='magma',
        xticklabels=['No Fraude', 'Fraude'],
        yticklabels=['No Fraude', 'Fraude']
    )

    plt.title(f'Matriz de Confusión - {nombre}')
    plt.xlabel('Predicción')
    plt.ylabel('Real')

    plt.show()


    resultados.append({
        'Modelo': nombre,
        'ROC-AUC': roc_auc,
        'PR-AUC': pr_auc
    })


resultados_df = pd.DataFrame(resultados)

resultados_df = resultados_df.sort_values(
    by='PR-AUC',
    ascending=False
)

print("\n\n=== COMPARACION FINAL ===\n")

display(resultados_df)


### Paso 6: Evaluacion de Modelos con PyCaret

In [ ]:
df_pc = df.copy()

setup(
    data=df_pc,
    target='FraudFound_P',
    session_id=42,
    train_size=0.8,
    fold=5,
    fold_strategy='stratifiedkfold',
    fix_imbalance=True,
    preprocess=True,
    normalize=True,
    verbose=False,
    html=False
)

_ = compare_models(sort='Recall', turbo=True)
pull()[['Model','Accuracy','AUC','Recall','Prec.','F1','Kappa','MCC']]


In [ ]:
res = metricas.copy()

mejor_recall = res.sort_values(['Recall','AUC'], ascending=False).iloc[0]
mejor_auc = res.sort_values(['AUC','Recall'], ascending=False).iloc[0]
mejor_f1 = res.sort_values(['F1','Recall'], ascending=False).iloc[0]

print('Analisis de resultados (PyCaret)')
print('-' * 50)
print(f"Mejor para detectar fraudes (Recall): {mejor_recall['Model']} | Recall={mejor_recall['Recall']:.4f} | Prec={mejor_recall['Prec.']:.4f}")
print(f"Mejor discriminacion global (AUC): {mejor_auc['Model']} | AUC={mejor_auc['AUC']:.4f} | Recall={mejor_auc['Recall']:.4f}")
print(f"Mejor balance precision-recall (F1): {mejor_f1['Model']} | F1={mejor_f1['F1']:.4f} | Recall={mejor_f1['Recall']:.4f} | Prec={mejor_f1['Prec.']:.4f}")

print('\nTop 5 por Recall:')
display(res.sort_values(['Recall','AUC'], ascending=False).head(5))
